# Olfactory Receptor activation
## https://academic.oup.com/nar/article/52/D1/D1370/7327067
## https://m2or.chemsensim.fr/

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ENRICHED_CSV = Path("Data/M2OR_pair_processed.csv")
OUT_FULL     = Path("or_activation_all.csv")
OUT_SAMPLE   = Path("or_activation_80.csv")
RANDOM_SEED  = 42

def read_auto_csv(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep=";", dtype=str, engine="python")
        if df.shape[1] == 1:
            raise ValueError
        return df
    except Exception:
        return pd.read_csv(path, sep=",", dtype=str, engine="python")

def uniq_in_order(values):
    seen, out = set(), []
    for v in values:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out

def assemble_rows(df, sep=", "):
    rows = []
    for smiles, g in df.groupby("smiles", sort=False):
        name_rep = g["name"].iloc[0]
        options  = uniq_in_order(g["gene_id"].tolist())
        answers  = uniq_in_order(g.loc[g["responsive"] == 1, "gene_id"].tolist())

        options_str = sep.join(options)
        answer_str  = sep.join(answers)

        prompt1 = (
            f"Among the following [{options_str}] olfactory receptor gene IDs choose all "
            f"olfactory receptors that molecule [{smiles}] activates. Do not write any comments."
        )
        prompt2 = (
            f"Among the following [{options_str}] olfactory receptor gene IDs choose all "
            f"olfactory receptors that molecule [{name_rep}] activates. Do not write any comments."
        )

        rows.append({
            "compound.name_1": name_rep,
            "SMILES_1": smiles,
            "compound.name_2": "",
            "SMILES_2": "",
            "OPTIONS": options_str,
            "question_category": "or_activation",
            "prompt.1": prompt1,
            "prompt.2": prompt2,
            "answer": answer_str,
            "_n_options": len(options),
            "_n_answers": len(answers),
            "other_info": "",
        })
    return pd.DataFrame(rows)

def main():
    df = read_auto_csv(ENRICHED_CSV)

    required = {"responsive", "smiles", "mixture", "name", "gene_id", "species"}
    missing = required - set(df.columns)
    if missing:
        raise SystemExit(f"Missing required columns: {sorted(missing)}")

    df = df[list(required)].copy()
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip()

    mask_species = df["species"].str.lower() == "homo sapiens"
    mask_mixture = df["mixture"].str.lower().isin({"mono", "sum of isomers"})
    df = df.loc[mask_species & mask_mixture].copy()
    if df.empty:
        raise SystemExit("No rows after species & mixture filters.")

    df["responsive"] = pd.to_numeric(df["responsive"], errors="coerce").fillna(0).astype(int)

    full_df = assemble_rows(df, sep=", ")
    if full_df.empty:
        raise SystemExit("No questions produced for full file.")
    full_df.insert(0, "question_ID", np.arange(1, len(full_df) + 1))
    full_df.drop(columns=["_n_options", "_n_answers"]).to_csv(OUT_FULL, index=False)
    print(f"Saved full set: {OUT_FULL.resolve()} ({len(full_df)} rows)")

    semi_df = assemble_rows(df, sep=";")
    semi_df = semi_df.query("_n_answers >= 1 and _n_options >= 4 and _n_options <= 10").copy()
    if semi_df.empty:
        print("No rows meeting (answers ≥1 and 4–10 options); skipping sample.")
        return

    rng = np.random.default_rng(RANDOM_SEED)
    k = min(80, len(semi_df))
    sample_idx = rng.choice(semi_df.index.to_numpy(), size=k, replace=False)
    sample = semi_df.loc[sorted(sample_idx)].copy()

    sample.insert(0, "question_ID", np.arange(1, len(sample) + 1))
    sample.drop(columns=["_n_options", "_n_answers"]).to_csv(OUT_SAMPLE, index=False)
    print(f"Saved sample set: {OUT_SAMPLE.resolve()} ({len(sample)} rows)")

if __name__ == "__main__":
    main()


Saved full set: /Users/vs479/Desktop/Desktop/HNL/OlfactoryInteligence_Benchmark/6-Hard_1/or_activation_all.csv (663 rows)
Saved sample set: /Users/vs479/Desktop/Desktop/HNL/OlfactoryInteligence_Benchmark/6-Hard_1/or_activation_80.csv (80 rows)
